# AT1 — Perceptron e KNN em Prática

**Disciplina:** Aprendizagem de Máquina
**Aluno:** Carlos Eduardo Campos Takeshita

Este notebook resolve os três desafios da atividade usando apenas **NumPy**:

1. **Desafio 1** — Classificação binária com Perceptron treinável (triagem de fraude)
2. **Desafio 2** — Predição de risco de churn com classificador KNN
3. **Desafio 3** — Recomendação de servidores cloud por similaridade espacial

Todas as células executam em ordem, do início ao fim (*Restart Kernel and Run All Cells*).

In [118]:
import numpy as np

# Deixa a impressão dos arrays mais legível (4 casas decimais, sem notação científica)
np.set_printoptions(precision=4, suppress=True)

---
## Desafio 1 — Classificação Binária com Perceptron Treinável

**Problema:** triagem de transações com risco de fraude a partir de duas métricas:

- $x_1$: valor da transação normalizado (0.0 a 10.0)
- $x_2$: frequência de operações nos últimos 5 minutos

**Saída:** `0` = Transação Legítima, `1` = Transação Suspeita.

O Perceptron calcula a combinação linear $z = w \cdot x + b$ e aplica a função degrau:
se $z \ge 0$ a classe é 1, senão é 0.

### 1.1 Dados de treinamento

In [119]:
# Matriz de treino: [Valor Normalizado (0.0 a 10.0), Frequência de Operações]
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

# Rótulos das classes (0: Legítima, 1: Suspeita)
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("Formato de X:", X_treino_fraude.shape)
print("Formato de y:", y_treino_fraude.shape)

Formato de X: (8, 2)
Formato de y: (8,)


### 1.2 Função de treinamento (regra de Rosenblatt)

Para cada amostra, o modelo faz a predição e compara com o rótulo real.
Só há ajuste quando a predição está errada:

$$w \leftarrow w + \eta \cdot (y - \hat{y}) \cdot x \qquad b \leftarrow b + \eta \cdot (y - \hat{y})$$

- Pesos e viés começam em **zero**, como pede o enunciado.
- Se uma época inteira passa sem nenhum erro, o modelo já separou as classes e o treino para antes do limite de épocas.

In [120]:
def funcao_degrau(z):
    """Função de ativação: retorna 1 se z >= 0, caso contrário 0."""
    return np.where(z >= 0, 1, 0)


def treinar_perceptron(X, y, taxa_aprendizado, epocas):
    """Treina um Perceptron com a regra de Rosenblatt e retorna (w, b)."""
    n_atributos = X.shape[1]

    # Inicialização exigida no enunciado: pesos e viés iguais a zero
    w = np.zeros(n_atributos)
    b = 0.0

    for epoca in range(1, epocas + 1):
        erros_na_epoca = 0

        for x_i, y_i in zip(X, y):
            z = np.dot(w, x_i) + b          # combinação linear
            y_pred = funcao_degrau(z)       # predição (0 ou 1)
            erro = y_i - y_pred             # +1, -1 ou 0

            # Ajuste somente para amostras classificadas errado
            if erro != 0:
                w = w + taxa_aprendizado * erro * x_i
                b = b + taxa_aprendizado * erro
                erros_na_epoca += 1

        # Época sem erro: os dados já estão separados, pode parar
        if erros_na_epoca == 0:
            print(f"Convergiu na época {epoca} (nenhum erro de predição).")
            break
    else:
        print(f"Atingiu o limite de {epocas} épocas sem zerar os erros.")

    return w, b

### 1.3 Função de inferência

Recebe uma amostra (ou uma matriz de amostras) e os parâmetros calibrados,
calcula a combinação linear e aplica a função degrau.

In [121]:
def prever_perceptron(x, w, b):
    """Retorna a classe prevista (0 ou 1) para uma amostra ou para várias."""
    z = np.dot(x, w) + b
    return funcao_degrau(z)

### 1.4 Treinamento e análise

Hiperparâmetros escolhidos: taxa de aprendizado $\eta = 0.1$ e limite de 100 épocas.

In [122]:
TAXA_APRENDIZADO = 0.1
EPOCAS = 100

w_fraude, b_fraude = treinar_perceptron(
    X_treino_fraude, y_treino_fraude, TAXA_APRENDIZADO, EPOCAS
)

print("Pesos finais (w):", w_fraude)
print(f"Viés final (b):   {b_fraude:.4f}")

# Conferência: o modelo acerta todas as amostras de treino?
pred_treino = prever_perceptron(X_treino_fraude, w_fraude, b_fraude)
acuracia = np.mean(pred_treino == y_treino_fraude)
print("Predições no treino:", pred_treino)
print("Rótulos reais:      ", y_treino_fraude)
print(f"Acurácia no treino:  {acuracia:.0%}")

Convergiu na época 6 (nenhum erro de predição).
Pesos finais (w): [0.05 0.1 ]
Viés final (b):   -0.8000
Predições no treino: [0 0 0 0 1 1 1 1]
Rótulos reais:       [0 0 0 0 1 1 1 1]
Acurácia no treino:  100%


### 1.5 Classificação das novas transações

In [123]:
rotulos_fraude = {0: "Transação Legítima", 1: "Transação Suspeita (Risco de Fraude)"}

novas_transacoes = {
    "Transação A": np.array([2.5, 2.0]),
    "Transação B": np.array([8.0, 6.5]),
}

for nome, x in novas_transacoes.items():
    z = np.dot(w_fraude, x) + b_fraude
    classe = int(prever_perceptron(x, w_fraude, b_fraude))
    print(f"{nome} {x} -> z = {z:+.4f} -> classe {classe}: {rotulos_fraude[classe]}")

Transação A [2.5 2. ] -> z = -0.4750 -> classe 0: Transação Legítima
Transação B [8.  6.5] -> z = +0.2500 -> classe 1: Transação Suspeita (Risco de Fraude)


**Leitura do resultado:** o sinal de $z$ indica de que lado da fronteira de decisão a transação caiu.
A Transação A (valor baixo e poucas operações) tem $z$ negativo e fica do lado das legítimas, junto
das amostras de treino da classe 0. A Transação B (valor alto e muitas operações em 5 minutos) tem
$z$ positivo e cai na região das suspeitas.

Como os pesos e o viés começam em zero, a taxa de aprendizado só muda a escala de $w$ e $b$:
a reta que separa as classes é a mesma, então o diagnóstico não depende do valor de $\eta$.

---
## Desafio 2 — Predição de Risco de Churn com Classificador KNN

**Problema:** antecipar o cancelamento de clientes de um SaaS a partir de:

- $x_1$: dias de inatividade nos últimos 30 dias
- $x_2$: chamados críticos em aberto no mês

**Saída:** `0` = Baixo Risco, `1` = Alto Risco.

O KNN não tem fase de treino: ele guarda a base histórica e, para um novo cliente,
olha os $K$ vizinhos mais próximos e decide pela classe mais frequente entre eles.

### 2.1 Dados de treinamento

In [124]:
# Matriz de treino: [Dias de Inatividade, Chamados Críticos]
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

# Rótulos (0: Baixo Risco, 1: Alto Risco)
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

### 2.2 Cálculo vetorizado das distâncias

`X - ponto` usa *broadcasting*: o ponto é subtraído de todas as linhas de uma vez,
sem laço manual. Depois, somando ao longo das colunas (`axis=1`), sai uma distância por amostra.

- **Euclidiana:** $\sqrt{\sum (x_i - p_i)^2}$ — linha reta
- **Manhattan:** $\sum |x_i - p_i|$ — soma das diferenças em cada eixo

In [125]:
def calcular_distancias(X, ponto, metrica="euclidiana"):
    """Distância de `ponto` até todas as linhas de X, de forma vetorizada."""
    diferencas = X - ponto  # broadcasting: subtrai o ponto de todas as linhas

    if metrica == "euclidiana":
        return np.sqrt(np.sum(diferencas ** 2, axis=1))
    if metrica == "manhattan":
        return np.sum(np.abs(diferencas), axis=1)

    raise ValueError("Métrica inválida. Use 'euclidiana' ou 'manhattan'.")

### 2.3 Função de classificação KNN

1. Calcula a distância do ponto até todas as amostras.
2. Ordena os índices da menor para a maior distância e pega os $K$ primeiros.
3. Conta os votos das classes desses vizinhos e devolve a mais votada.

In [126]:
def knn_classificar(X, y, ponto, k, metrica="euclidiana"):
    """Classifica `ponto` por maioria de votos entre os k vizinhos mais próximos.

    Retorna (classe_predita, indices_dos_vizinhos, distancias_dos_vizinhos).
    """
    distancias = calcular_distancias(X, ponto, metrica)

    # Índices ordenados pela distância; ficam só os k mais próximos
    indices_vizinhos = np.argsort(distancias, kind="stable")[:k]

    # Maioria de votos: bincount conta quantos vizinhos há em cada classe
    votos = np.bincount(y[indices_vizinhos])
    classe_predita = int(np.argmax(votos))

    return classe_predita, indices_vizinhos, distancias[indices_vizinhos]

### 2.4 Execução e diagnóstico

Cada cliente é avaliado com o seu valor de $K$, nas duas métricas de distância.

In [127]:
rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}

# Clientes sob avaliação: (nome, ponto, K)
K_CLIENTE_1 = 3
K_CLIENTE_2 = 3

clientes = [
    ("Cliente 1", np.array([4.0, 1.0]), K_CLIENTE_1),
    ("Cliente 2", np.array([17.0, 5.0]), K_CLIENTE_2),
]

for nome, ponto, k in clientes:
    print(f"{nome} {ponto} com K = {k}")
    for metrica in ("euclidiana", "manhattan"):
        classe, indices, distancias = knn_classificar(
            X_treino_churn, y_treino_churn, ponto, k, metrica
        )
        print(f"  Métrica {metrica}:")
        print(f"    Classe predita:        {rotulos_churn[classe]}")
        print(f"    Índices dos vizinhos:  {indices}")
        print(f"    Distâncias:            {distancias}")
        print(f"    Classes dos vizinhos:  {y_treino_churn[indices]}")
    print()

Cliente 1 [4. 1.] com K = 3
  Métrica euclidiana:
    Classe predita:        Baixo Risco
    Índices dos vizinhos:  [2 1 0]
    Distâncias:            [1.     1.4142 2.    ]
    Classes dos vizinhos:  [0 0 0]
  Métrica manhattan:
    Classe predita:        Baixo Risco
    Índices dos vizinhos:  [2 0 1]
    Distâncias:            [1. 2. 2.]
    Classes dos vizinhos:  [0 0 0]

Cliente 2 [17.  5.] com K = 3
  Métrica euclidiana:
    Classe predita:        Alto Risco
    Índices dos vizinhos:  [5 4 6]
    Distâncias:            [1.     2.2361 3.1623]
    Classes dos vizinhos:  [1 1 1]
  Métrica manhattan:
    Classe predita:        Alto Risco
    Índices dos vizinhos:  [5 4 6]
    Distâncias:            [1. 3. 4.]
    Classes dos vizinhos:  [1 1 1]



**Leitura do resultado:** o Cliente 1 (4 dias de inatividade e 1 chamado) tem os três vizinhos
mais próximos na classe 0, então é **Baixo Risco**. O Cliente 2 (17 dias e 5 chamados) tem os três
vizinhos na classe 1, então é **Alto Risco**. Nas duas métricas a classe final é a mesma; o que
muda são os valores das distâncias e, no Cliente 1, a ordem entre os vizinhos 0 e 1, que empatam
na distância Manhattan.

---
## Desafio 3 — Recomendação de Servidores Cloud por Similaridade Espacial

**Problema:** dado o hardware que uma aplicação precisa, recomendar as máquinas virtuais do
catálogo mais parecidas no espaço tridimensional:

- $x_1$: vCPUs
- $x_2$: memória RAM (GB)
- $x_3$: armazenamento SSD NVMe (GB)

Aqui **não há rótulos de classe**. É a mesma ideia de vizinhança do KNN, mas em vez de votar
em uma classe, o resultado é o próprio ranking dos vizinhos mais próximos.

### 3.1 Catálogo de instâncias

In [128]:
# Catálogo: [vCPUs, Memória RAM (GB), Armazenamento SSD (GB)]
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

### 3.2 Função de recomendação

Reaproveita `calcular_distancias` do Desafio 2: calcula a distância da demanda até todos os
servidores de uma vez e devolve os índices dos $K$ mais próximos, já ordenados.

In [129]:
def recomendar_servidores(catalogo, perfil, k, metrica="euclidiana"):
    """Retorna (indices, distancias) dos k servidores mais próximos do perfil."""
    distancias = calcular_distancias(catalogo, perfil, metrica)

    # Ordena do mais parecido para o menos parecido e fica com os k primeiros
    indices_ordenados = np.argsort(distancias, kind="stable")[:k]

    return indices_ordenados, distancias[indices_ordenados]

### 3.3 Ranking de recomendações

In [130]:
# Perfil de hardware demandado: [vCPUs, RAM (GB), SSD (GB)]
perfil_demandado = np.array([12.0, 28.0, 450.0])
K_RECOMENDACOES = 2

indices, distancias = recomendar_servidores(
    catalogo_servidores, perfil_demandado, K_RECOMENDACOES, metrica="euclidiana"
)

vcpus, ram, ssd = perfil_demandado
print("=" * 62)
print("RELATÓRIO DE RECOMENDAÇÃO DE SERVIDORES")
print("=" * 62)
print(f"Demanda: {vcpus:.0f} vCPUs | {ram:.0f} GB RAM | {ssd:.0f} GB SSD")
print(f"Recomendações solicitadas (K): {K_RECOMENDACOES}")
print("-" * 62)

for posicao, (indice, distancia) in enumerate(zip(indices, distancias), start=1):
    vcpus, ram, ssd = catalogo_servidores[indice]
    print(f"{posicao}º lugar: {nomes_servidores[indice]}")
    print(f"   Especificações: {vcpus:.0f} vCPUs | {ram:.0f} GB RAM | {ssd:.0f} GB SSD")
    print(f"   Distância euclidiana: {distancia:.4f}")
    print("-" * 62)

RELATÓRIO DE RECOMENDAÇÃO DE SERVIDORES
Demanda: 12 vCPUs | 28 GB RAM | 450 GB SSD
Recomendações solicitadas (K): 2
--------------------------------------------------------------
1º lugar: Database Enterprise
   Especificações: 16 vCPUs | 32 GB RAM | 500 GB SSD
   Distância euclidiana: 50.3190
--------------------------------------------------------------
2º lugar: Medium Backend & Cache
   Especificações: 8 vCPUs | 16 GB RAM | 250 GB SSD
   Distância euclidiana: 200.3996
--------------------------------------------------------------


**Leitura do resultado:** a *Database Enterprise* (16 vCPUs, 32 GB, 500 GB) é a máquina mais
próxima da demanda e já atende aos três requisitos. A 2ª colocada, *Medium Backend & Cache*, é a
próxima mais parecida geometricamente, mas fica abaixo do pedido em vCPUs, RAM e SSD — a distância
mede semelhança, não garante que a máquina seja suficiente.

A distância é dominada pelo SSD, que está em uma escala muito maior (centenas de GB) do que vCPUs
e RAM (dezenas). Em um sistema real, normalizar os atributos antes de calcular a distância daria
peso equilibrado às três dimensões.

---
## Conclusão

- **Perceptron:** aprende uma fronteira linear ajustando pesos e viés só quando erra; como os dados são linearmente separáveis, o treino converge.
- **KNN:** não aprende parâmetros; classifica pela maioria dos vizinhos mais próximos, e o resultado depende de $K$ e da métrica.
- **Recomendação por similaridade:** mesma base do KNN, sem rótulos — o ranking de distâncias é a própria resposta.